# Spline SPF Injection-Recovery Test

This notebook validates that the `InterpolatedUnivariateSpline_SPF` formalism can accurately recover a known Henyey-Greenstein (HG) scattering phase function from a synthetic disk image.

**Strategy:**
1. *Inject* — generate a synthetic disk image using a known HG SPF (g = 0.3).
2. *Recover* — fit the same disk using the spline SPF, holding all geometry fixed, optimising the knot values and `flux_scaling` jointly.
3. *Compare* — plot the recovered spline SPF against the normalised true HG (`HG(cos φ) / HG(0)`) and quantify the agreement.

We repeat the test for **four inclinations** (30°, 50°, 70°, 80°). Higher inclination → wider range of sampled scattering angles → better recovery.

**Normalisation:** The spline is normalised to 1 at 90° (cos φ = 0), so SPF comparisons are always against the normalised HG. The absolute flux amplitude is folded into `flux_scaling`, which is also a free parameter in the fit.

**Initial `flux_scaling`:** Because the spline is normalised to 1 at 90°, the correct starting value is `FLUX × HG(90°)`. In this injection-recovery test we know the truth and can use this directly. With real data a user would supply a physically motivated estimate (e.g. derived from stellar brightness, grain albedo, and system geometry).

**Knot placement:** uniformly spaced across the full cos φ ∈ [−1, 1] (0°–180°). Only the disk's probed range is constrained by data; knots outside it remain near their flat initial value.

In [ ]:
import os
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.5"

from grater_jax.disk_model.SLD_ojax import ScatteredLightDisk
from grater_jax.disk_model.SLD_utils import (
    DustEllipticalDistribution2PowerLaws,
    HenyeyGreenstein_SPF,
    InterpolatedUnivariateSpline_SPF,
    recommended_num_knots,
)
from grater_jax.disk_model.objective_functions import objective_model, Parameter_Index
from grater_jax.optimization.optimize_framework import Optimizer

PLOT_DIR = "spline_recovery_plots"
os.makedirs(PLOT_DIR, exist_ok=True)
print("Output directory:", PLOT_DIR)

## True (injected) parameters

In [ ]:
G_TRUE    = 0.3
hg_at_90  = float((1.0 / (4.0 * np.pi)) * (1 - G_TRUE**2) / (1 + G_TRUE**2)**1.5)
print(f"HG(cos phi=0, g={G_TRUE}) = {hg_at_90:.6f}")

NX, NY    = 140, 140
FLUX      = 1e6
NUM_KNOTS = 6     # 6 knots is usually enough for smooth SPFs; reduces parameter count
FIT_ITERS = 2000

# Assumed asymmetry parameter for spline SPF initialisation.
# Starting from a flat SPF risks converging to a local minimum, particularly
# at low inclinations where the probed angular range is narrow and the gradient
# signal is weak. Initialising from an HG shape with INIT_G provides a
# physically motivated starting point (most debris disk SPFs are forward-
# scattering). The optimiser refines from there. INIT_G does not need to match
# the true SPF — it is an assumption, not a constraint.
INIT_G = 0.5

BASE_MISC = Parameter_Index.misc_params.copy()
BASE_MISC.update({
    "nx": NX, "ny": NY,
    "distance": 50.0,
    "pxInArcsec": 0.01225,
    "halfNbSlices": 25,
    "flux_scaling": FLUX,
})

BASE_DISK = Parameter_Index.disk_params.copy()
BASE_DISK.update({
    "sma": 46.0, "alpha_in": 5.0, "alpha_out": -5.0,
    "ksi0": 1.0, "gamma": 2.0, "beta": 1.0,
    "e": 0.0, "dens_at_r0": 1.0,
    "position_angle": 0.0, "omega": 0.0,
    "x_center": NX / 2, "y_center": NY / 2,
    "halfNbSlices": 25,
})

HG_SPF_PARAMS = HenyeyGreenstein_SPF.params.copy()
HG_SPF_PARAMS["g"] = G_TRUE

INCLINATIONS = [30.0, 50.0, 70.0, 80.0]

## Helpers

In [ ]:
def hg_normalized(cos_phi, g):
    """HG phase function normalised to 1 at cos_phi=0 (90 deg)."""
    raw  = (1.0 / (4.0 * np.pi)) * (1 - g**2) / (1 + g**2 - 2*g*cos_phi)**1.5
    norm = (1.0 / (4.0 * np.pi)) * (1 - g**2) / (1 + g**2)**1.5
    return raw / norm

def probed_cosphi_range(incl_deg):
    """cos(phi) range sampled by a disk at a given inclination.
    Scattering angles range from ~(90-incl) to ~(90+incl) degrees,
    so cos(phi) spans [-sin(incl), +sin(incl)].
    """
    s = np.sin(np.radians(incl_deg))
    return -s, s  # (cp_back, cp_fwd)

assert abs(hg_normalized(0.0, G_TRUE) - 1.0) < 1e-12

cos_grid = np.linspace(-1.0, 1.0, 500)
angles   = np.degrees(np.arccos(cos_grid))

plt.figure(figsize=(6, 3))
plt.plot(angles, hg_normalized(cos_grid, G_TRUE), "k-", lw=2)
plt.axhline(1.0, color="grey", ls="--", lw=0.8)
plt.axvline(90,  color="grey", ls="--", lw=0.8)
plt.xlabel("Scattering angle (deg)")
plt.ylabel("Normalised SPF")
plt.title(f"True HG SPF (g={G_TRUE}), normalised at 90 deg")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "true_hg_spf.png"), dpi=150)
plt.show()

## Injection-recovery loop

**Fit strategy:** joint optimisation of `knot_values` + `flux_scaling` using L-BFGS-B with analytic JAX gradients.  If the first run terminates early (line-search failure), a warm restart resets the quasi-Newton memory and continues from where it stopped.

**Inclination-dependent knot placement:** rather than spanning the full cos φ ∈ [−1, 1], knots are placed only within the scattering angles actually probed by the disk (≈ [90°−*i*, 90°+*i*]), with a small buffer beyond the geometric boundary.  The knot count scales linearly with the probed angular window via `recommended_num_knots`, with a minimum of 4 free knots — empirically the minimum needed to avoid local minima.

**SPF initialisation:** knot values are initialised from a Henyey-Greenstein phase function with assumed asymmetry `INIT_G` (set in the constants cell).  A flat initialisation risks converging to an image-consistent but SPF-inconsistent local minimum at low inclinations where the gradient signal is weak.  `INIT_G` is an assumption about the degree of forward scattering, not a constraint — the optimiser adjusts freely from there.

**Known lower limit:** inclinations below roughly 28° do not probe a wide enough scattering-angle range to constrain the SPF shape reliably, regardless of knot count or initialisation.  Recovery fails below this threshold because multiple SPF shapes produce indistinguishable disk images.

In [ ]:
results = {}

JOINT_KEYS      = ["knot_values", "flux_scaling"]
JOINT_LOGSCALED = ["knot_values", "flux_scaling"]
JOINT_ARRAYS    = ["knot_values"]

# Small buffer beyond the geometric probed range so knots aren't placed
# exactly at the boundary — gives the spline a bit of room at the edges.
KNOT_BOUND_BUFFER = 0.1   # in cos_phi units

for incl in INCLINATIONS:
    print(f"\n{'='*55}\nInclination = {incl} deg")

    # ── 0. Compute bounds and scale num_knots to probed window ───────────────
    cp_back, cp_fwd = probed_cosphi_range(incl)
    nk         = recommended_num_knots(incl, NUM_KNOTS, boundary_buffer=KNOT_BOUND_BUFFER)
    fwd_bound  = float(np.clip(cp_fwd  + KNOT_BOUND_BUFFER, -1.0, 1.0))
    back_bound = float(np.clip(cp_back - KNOT_BOUND_BUFFER, -1.0, 1.0))

    # ── 1. Build truth image ──────────────────────────────────────────────────
    disk_params = BASE_DISK.copy()
    disk_params["inclination"] = incl

    truth_misc = BASE_MISC.copy()
    truth_misc["flux_scaling"] = FLUX

    truth_opt = Optimizer(
        ScatteredLightDisk, DustEllipticalDistribution2PowerLaws,
        HenyeyGreenstein_SPF, None,
        disk_params, HG_SPF_PARAMS, None, truth_misc,
    )
    truth_img = np.array(truth_opt.get_model())
    err_map   = np.sqrt(np.abs(truth_img)) * np.sqrt(truth_img.max()) * 0.01
    print(f'  Truth image: max={truth_img.max():.4e}  err_map: {err_map.max():.4e}')

    # ── 2. Set up spline optimizer with inclination-dependent knot bounds ─────
    # (cp_back, cp_fwd, fwd_bound, back_bound, nk already set in step 0)
    spf_params = InterpolatedUnivariateSpline_SPF.params.copy()
    spf_params["num_knots"]          = nk
    spf_params["knot_values"]        = jnp.ones(nk)
    spf_params["forwardscatt_bound"] = fwd_bound
    spf_params["backscatt_bound"]    = back_bound

    fit_misc = BASE_MISC.copy()
    fit_misc["flux_scaling"]   = FLUX * hg_at_90

    opt = Optimizer(
        ScatteredLightDisk, DustEllipticalDistribution2PowerLaws,
        InterpolatedUnivariateSpline_SPF, None,
        disk_params, spf_params, None, fit_misc,
    )

    nk = opt.spf_params["num_knots"]
    knot_cos = np.array(InterpolatedUnivariateSpline_SPF.get_knots(opt.spf_params))
    knot_angles_deg = np.degrees(np.arccos(np.clip(knot_cos, -1, 1)))
    print(f'  Knot bounds: {np.degrees(np.arccos(fwd_bound)):.1f}–{np.degrees(np.arccos(back_bound)):.1f} deg  '
          f'(probed {np.degrees(np.arccos(cp_fwd)):.1f}–{np.degrees(np.arccos(cp_back)):.1f} deg)')
    print(f'  Knot angles: {np.round(knot_angles_deg, 1)}')

    # ── 3. Initialise from assumed HG shape ───────────────────────────────────
    # Starting from a flat SPF (all ones) risks converging to a local minimum
    # at low inclinations where the probed range is narrow. Initialising the
    # free knot values from an HG shape with INIT_G avoids this. INIT_G is an
    # assumption, not a constraint — the optimiser adjusts freely from there.
    n_left_init  = nk // 2
    free_knot_cos = np.concatenate([knot_cos[:n_left_init], knot_cos[n_left_init + 1:]])
    init_kv = hg_normalized(free_knot_cos, INIT_G)
    opt.spf_params["knot_values"]   = jnp.array(init_kv)
    opt.misc_params["flux_scaling"] = FLUX * hg_at_90
    joint_bounds = ([np.full(nk, 1e-6), 1e2], [np.full(nk, 1e3), 1e12])
    print(f'  flux_scaling init = {opt.misc_params["flux_scaling"]:.4e}  (= FLUX * HG(90 deg))')

    # ── 4. Joint fit with optional warm restarts ──────────────────────────────
    nit_total = 0
    for pass_num in range(1, 4):
        soln = opt.scipy_bounded_optimize(
            fit_keys=JOINT_KEYS,
            fit_bounds=joint_bounds,
            logscaled_params=JOINT_LOGSCALED,
            array_params=JOINT_ARRAYS,
            target_image=truth_img,
            err_map=err_map,
            use_grad=True, iters=FIT_ITERS, ftol=1e-14, gtol=1e-14,
        )
        nit_total += soln.nit
        status = "converged" if soln.success else soln.message[:40].strip()
        print(f'  Pass {pass_num}: nit={soln.nit}  LL={opt.log_likelihood(truth_img, err_map):.4f}  [{status}]')
        if soln.success:
            break

    print(f'  Total nit={nit_total}  flux_scaling={opt.misc_params["flux_scaling"]:.4e}  (expected {FLUX*hg_at_90:.4e})')

    fitted_img = np.array(opt.get_model())

    # ── 5. Evaluate recovered SPF ─────────────────────────────────────────────
    spline_model = InterpolatedUnivariateSpline_SPF.pack_pars(
        opt.spf_params["knot_values"],
        knots=InterpolatedUnivariateSpline_SPF.get_knots(opt.spf_params)
    )
    recovered_spf = np.array(
        InterpolatedUnivariateSpline_SPF.compute_phase_function_from_cosphi(
            spline_model, jnp.array(cos_grid)
        )
    )
    true_spf_norm = hg_normalized(cos_grid, G_TRUE)

    in_range = (cos_grid >= cp_back) & (cos_grid <= cp_fwd)
    frac_err = np.abs(recovered_spf[in_range] - true_spf_norm[in_range]) / (np.abs(true_spf_norm[in_range]) + 1e-30)
    max_frac_err = float(frac_err.max())
    print(f'  Max fractional SPF error (probed range): {max_frac_err*100:.2f}%')

    results[incl] = dict(
        truth_img=truth_img, fitted_img=fitted_img, err_map=err_map,
        recovered_spf=recovered_spf, true_spf_norm=true_spf_norm,
        knot_cos=knot_cos, knot_values=np.array(opt.spf_params["knot_values"]),
        flux_scaling=float(opt.misc_params["flux_scaling"]),
        spline_model=spline_model,
        cp_fwd=cp_fwd, cp_back=cp_back,
        fwd_bound=fwd_bound, back_bound=back_bound,
        in_range=in_range,
        max_frac_err=max_frac_err, nit_total=nit_total,
    )

    # ── 6. Disk image comparison ──────────────────────────────────────────────
    residual = truth_img - fitted_img
    vmin = np.percentile(truth_img[truth_img > 0], 5)
    vmax = truth_img.max()
    res_lim = np.max(np.abs(residual))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Inclination = {incl} deg", fontsize=13)
    im0 = axes[0].imshow(truth_img,  cmap="inferno", origin="lower", vmin=vmin, vmax=vmax)
    axes[0].set_title("Truth (HG SPF)")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    im1 = axes[1].imshow(fitted_img, cmap="inferno", origin="lower", vmin=vmin, vmax=vmax)
    axes[1].set_title("Fitted (Spline SPF)")
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    im2 = axes[2].imshow(residual,   cmap="seismic", origin="lower", vmin=-res_lim, vmax=res_lim)
    axes[2].set_title("Residual (Truth - Fit)")
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"disk_comparison_incl{int(incl):02d}.png"), dpi=150)
    plt.show()

    # ── 7. SPF comparison ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(angles, true_spf_norm, "k-",  lw=2,  label="True HG (normalised)")
    ax.plot(angles, recovered_spf, "C0-", lw=2,  label="Recovered spline")
    ang_fwd  = float(np.degrees(np.arccos(cp_fwd)))
    ang_back = float(np.degrees(np.arccos(cp_back)))
    ax.axvspan(ang_fwd, ang_back, alpha=0.12, color="C0",
               label=f"Probed range ({ang_fwd:.0f} to {ang_back:.0f} deg)")
    knot_spf_vals = np.array(
        InterpolatedUnivariateSpline_SPF.compute_phase_function_from_cosphi(
            spline_model, jnp.array(knot_cos)
        )
    )
    ax.scatter(knot_angles_deg, knot_spf_vals, color="C0", zorder=5, s=50, label="Knots")
    ax.axvline(90, color="grey", ls="--", lw=0.8)
    ax.axhline(1.0, color="grey", ls="--", lw=0.8)
    ax.set_xlabel("Scattering angle (deg)")
    ax.set_ylabel("Normalised SPF")
    ax.set_title(f"SPF Recovery  incl={incl} deg  max err={max_frac_err*100:.1f}%")
    ax.legend(fontsize=9)
    ax.set_xlim(0, 180)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"spf_comparison_incl{int(incl):02d}.png"), dpi=150)
    plt.show()

## Summary: SPF recovery across all inclinations

In [ ]:
colors = ["C0", "C1", "C2", "C3"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(angles, hg_normalized(cos_grid, G_TRUE), "k-", lw=2.5,
        label=f"True HG (g={G_TRUE})", zorder=10)

for incl, color in zip(INCLINATIONS, colors):
    r = results[incl]
    ax.plot(angles, r["recovered_spf"], "-", color=color, lw=1.8,
            label=f"Spline incl={int(incl)} deg  (err<={r['max_frac_err']*100:.1f}%)")
    ang_fwd  = float(np.degrees(np.arccos(r["cp_fwd"])))
    ang_back = float(np.degrees(np.arccos(r["cp_back"])))
    ax.axvspan(ang_fwd, ang_back, alpha=0.07, color=color)

ax.axvline(90,  color="grey", ls="--", lw=0.8)
ax.axhline(1.0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("Scattering angle (deg)")
ax.set_ylabel("Normalised SPF")
ax.set_title("SPF Injection-Recovery Summary")
ax.legend(fontsize=9)
ax.set_xlim(0, 180)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "spf_recovery_summary.png"), dpi=150)
plt.show()

print()
print(f"{'Inclination':>15}  {'Max SPF error (probed range)':>30}")
print("-" * 48)
for incl in INCLINATIONS:
    print(f"{incl:>14.0f} deg  {results[incl]['max_frac_err']*100:>27.2f}%")

## Fractional residual images

In [ ]:
fig, axes = plt.subplots(1, len(INCLINATIONS), figsize=(4*len(INCLINATIONS), 4))

for ax, incl in zip(axes, INCLINATIONS):
    r = results[incl]
    truth   = r["truth_img"]
    bright  = truth > 1e-3 * truth.max()
    frac_resid = np.zeros_like(truth)
    frac_resid[bright] = (truth[bright] - r["fitted_img"][bright]) / truth[bright]
    lim = np.percentile(np.abs(frac_resid[bright]), 99)
    im  = ax.imshow(frac_resid, cmap="seismic", origin="lower", vmin=-lim, vmax=lim)
    max_err_pct = np.abs(frac_resid[bright]).max() * 100
    ax.set_title(f"incl={int(incl)} deg  max|err|={max_err_pct:.1f}%")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="(T-F)/T")

fig.suptitle("Fractional residual (Truth - Fit) / Truth", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "fractional_residuals.png"), dpi=150)
plt.show()

## Interpretation

- **Higher inclination → better recovery.** An 80° disk samples scattering angles from ~10° to ~170°, giving the spline a well-constrained shape across almost the full range. A 30° disk only probes ~60°–120°; knots outside that window remain at their flat initial value and are unconstrained.
- **Recovered ** should converge to approximately  because the spline normalises away the 90° value and transfers it into .
- **Residual structure.** Systematic ring- or spoke-like patterns indicate the spline shape does not yet perfectly match the HG curve over the probed angles. More knots can help at the cost of the fit being more underdetermined outside the probed range.

---

## Full Joint Fit: SPF + Geometry

The previous section held disk geometry fixed, isolating the question of whether the
spline SPF can recover a known HG phase function. In practice a user fitting real data
would optimise the SPF **simultaneously** with at least the basic geometric parameters.

Here we repeat the injection-recovery with a joint fit over five parameters:

| Parameter | Role |
|---|---|
| `knot_values` | Spline SPF shape (6 free values) |
| `flux_scaling` | Absolute brightness scale |
| `inclination` | Disk tilt relative to sky plane |
| `position_angle` | Disk orientation on sky |
| `sma` | Ring semi-major axis |

The fit starts from a **perturbed** initial guess (5° off in inclination and PA, 10% off
in sma) rather than the true values, to simulate a realistic starting condition.
We run two inclinations (50° and 70°) as representative cases.

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────
GEOM_INCLINATIONS = [50.0, 70.0]   # subset of INCLINATIONS — joint fits are slower
GEOM_FIT_ITERS    = 3000

GEOM_KEYS      = ["knot_values", "flux_scaling", "inclination", "position_angle", "sma"]
GEOM_LOGSCALED = ["knot_values", "flux_scaling", "sma"]
GEOM_ARRAYS    = ["knot_values"]

# Perturb geometry from truth to simulate a realistic starting condition
INIT_INCL_OFFSET = 5.0   # deg
INIT_PA_OFFSET   = 5.0   # deg
INIT_SMA_FACTOR  = 1.1   # 10% high

geom_results = {}

for incl in GEOM_INCLINATIONS:
    print(f"\n{'='*55}\nInclination = {incl} deg  [joint SPF + geometry]")
    nk = NUM_KNOTS

    # ── 1. Build truth image ──────────────────────────────────────────────────
    true_disk_params = BASE_DISK.copy()
    true_disk_params["inclination"] = incl

    truth_misc = BASE_MISC.copy()
    truth_misc["flux_scaling"] = FLUX

    truth_opt = Optimizer(
        ScatteredLightDisk, DustEllipticalDistribution2PowerLaws,
        HenyeyGreenstein_SPF, None,
        true_disk_params, HG_SPF_PARAMS, None, truth_misc,
    )
    truth_img = np.array(truth_opt.get_model())
    err_map   = np.sqrt(np.abs(truth_img)) * np.sqrt(truth_img.max()) * 0.01

    # ── 2. Perturbed initial guess ────────────────────────────────────────────
    fit_disk_params = BASE_DISK.copy()
    fit_disk_params["inclination"]    = incl + INIT_INCL_OFFSET
    fit_disk_params["position_angle"] = BASE_DISK["position_angle"] + INIT_PA_OFFSET
    fit_disk_params["sma"]            = BASE_DISK["sma"] * INIT_SMA_FACTOR

    spf_params = InterpolatedUnivariateSpline_SPF.params.copy()
    spf_params["num_knots"]   = nk
    spf_params["knot_values"] = jnp.ones(nk)

    fit_misc = BASE_MISC.copy()
    fit_misc["flux_scaling"] = FLUX * hg_at_90

    opt = Optimizer(
        ScatteredLightDisk, DustEllipticalDistribution2PowerLaws,
        InterpolatedUnivariateSpline_SPF, None,
        fit_disk_params, spf_params, None, fit_misc,
    )

    print(f"  Init: incl={fit_disk_params['inclination']:.1f}  "
          f"PA={fit_disk_params['position_angle']:.1f}  "
          f"sma={fit_disk_params['sma']:.1f}")

    # ── 3. Bounds ─────────────────────────────────────────────────────────────
    geom_bounds = (
        [np.full(nk, 1e-6), 1e2,   0.0, -90.0, 20.0],
        [np.full(nk, 1e3),  1e12, 89.0,  90.0, 100.0],
    )

    # ── 4. Joint fit with warm restarts ───────────────────────────────────────
    nit_total = 0
    for pass_num in range(1, 4):
        soln = opt.scipy_bounded_optimize(
            fit_keys=GEOM_KEYS,
            fit_bounds=geom_bounds,
            logscaled_params=GEOM_LOGSCALED,
            array_params=GEOM_ARRAYS,
            target_image=truth_img,
            err_map=err_map,
            use_grad=True, iters=GEOM_FIT_ITERS, ftol=1e-14, gtol=1e-14,
        )
        nit_total += soln.nit
        status = "converged" if soln.success else soln.message[:40].strip()
        print(f"  Pass {pass_num}: nit={soln.nit}  "
              f"LL={opt.log_likelihood(truth_img, err_map):.4f}  [{status}]")
        if soln.success:
            break

    rec_incl = float(opt.disk_params["inclination"])
    rec_pa   = float(opt.disk_params["position_angle"])
    rec_sma  = float(opt.disk_params["sma"])
    rec_flux = float(opt.misc_params["flux_scaling"])

    print(f"  incl:  recovered={rec_incl:.2f}  truth={incl:.1f}  "
          f"err={abs(rec_incl - incl):.2f} deg")
    print(f"  PA:    recovered={rec_pa:.2f}  truth={BASE_DISK['position_angle']:.1f}  "
          f"err={abs(rec_pa - BASE_DISK['position_angle']):.2f} deg")
    print(f"  sma:   recovered={rec_sma:.2f}  truth={BASE_DISK['sma']:.1f}  "
          f"err={abs(rec_sma - BASE_DISK['sma']):.2f} AU")
    print(f"  flux:  recovered={rec_flux:.4e}  expected={FLUX*hg_at_90:.4e}")

    # ── 5. Recovered SPF ──────────────────────────────────────────────────────
    spline_model = InterpolatedUnivariateSpline_SPF.pack_pars(
        opt.spf_params["knot_values"],
        knots=InterpolatedUnivariateSpline_SPF.get_knots(opt.spf_params),
    )
    recovered_spf = np.array(
        InterpolatedUnivariateSpline_SPF.compute_phase_function_from_cosphi(
            spline_model, jnp.array(cos_grid)
        )
    )
    true_spf_norm = hg_normalized(cos_grid, G_TRUE)

    cp_back, cp_fwd = probed_cosphi_range(incl)
    in_range = (cos_grid >= cp_back) & (cos_grid <= cp_fwd)
    frac_err = (np.abs(recovered_spf[in_range] - true_spf_norm[in_range])
                / (np.abs(true_spf_norm[in_range]) + 1e-30))
    max_frac_err = float(frac_err.max())
    print(f"  Max SPF error (probed range): {max_frac_err*100:.2f}%")

    geom_results[incl] = dict(
        truth_img=truth_img,
        fitted_img=np.array(opt.get_model()),
        recovered_spf=recovered_spf,
        true_spf_norm=true_spf_norm,
        cp_fwd=cp_fwd, cp_back=cp_back,
        rec_incl=rec_incl, rec_pa=rec_pa, rec_sma=rec_sma,
        max_frac_err=max_frac_err,
    )

    # ── 6. SPF + geometry summary plot ────────────────────────────────────────
    ang_fwd  = float(np.degrees(np.arccos(cp_fwd)))
    ang_back = float(np.degrees(np.arccos(cp_back)))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"Joint fit  incl={incl} deg", fontsize=13)

    ax = axes[0]
    ax.plot(angles, true_spf_norm, "k-", lw=2, label="True HG (normalised)")
    ax.plot(angles, recovered_spf, "C1-", lw=2,
            label=f"Recovered spline (err={max_frac_err*100:.1f}%)")
    ax.axvspan(ang_fwd, ang_back, alpha=0.12, color="C1",
               label=f"Probed ({ang_fwd:.0f}–{ang_back:.0f} deg)")
    ax.axvline(90, color="grey", ls="--", lw=0.8)
    ax.axhline(1.0, color="grey", ls="--", lw=0.8)
    ax.set_xlabel("Scattering angle (deg)")
    ax.set_ylabel("Normalised SPF")
    ax.set_title("Scattering phase function")
    ax.legend(fontsize=9)
    ax.set_xlim(0, 180)

    ax = axes[1]
    ax.axis("off")
    table_data = [
        ["Parameter",          "Truth",                           "Recovered",         "Error"],
        ["inclination (deg)",  f"{incl:.1f}",                     f"{rec_incl:.2f}",   f"{abs(rec_incl - incl):.2f}"],
        ["position_angle (deg)", f"{BASE_DISK['position_angle']:.1f}", f"{rec_pa:.2f}", f"{abs(rec_pa - BASE_DISK['position_angle']):.2f}"],
        ["sma (AU)",           f"{BASE_DISK['sma']:.1f}",         f"{rec_sma:.2f}",    f"{abs(rec_sma - BASE_DISK['sma']):.2f}"],
        ["flux_scaling",       f"{FLUX*hg_at_90:.3e}",            f"{rec_flux:.3e}",   "—"],
    ]
    tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.2, 1.8)
    ax.set_title("Geometry recovery")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"joint_fit_incl{int(incl):02d}.png"), dpi=150)
    plt.show()

---

## MCMC Injection-Recovery

Now we repeat the injection-recovery using MCMC to sample the full posterior over `knot_values` and `flux_scaling`.  This verifies that the spline SPF works with Bayesian inference, not just point-estimation, and gives a sense of the posterior uncertainty on the recovered phase function.

**Strategy:**
1. Start from the scipy best-fit result for each inclination (already in `results`).
2. Run emcee with a small ball of walkers around that solution.
3. Extract posterior SPF samples from the chain and plot the ±1σ envelope against the true HG.

MCMC settings are kept modest for speed; increase `NWALKERS` / `NITER` for production.

In [ ]:
# ── MCMC settings ────────────────────────────────────────────────────────────
NWALKERS = 64     # emcee ensemble walkers
NITER    = 2000   # total production steps per walker (after burn-in)
BURNS    = 200    # burn-in steps to discard

MCMC_KEYS      = ["knot_values", "flux_scaling"]
MCMC_LOGSCALED = ["knot_values", "flux_scaling"]
MCMC_ARRAYS    = ["knot_values"]

# HDF5 backend files are written to this directory (one file per inclination)
MCMC_DIR = os.path.join(os.path.dirname(PLOT_DIR), "mcmc_backends")
os.makedirs(MCMC_DIR, exist_ok=True)

In [ ]:
mcmc_results = {}

for incl in INCLINATIONS:
    print(f"\n{'='*55}\nInclination = {incl} deg  [MCMC]")
    r  = results[incl]
    nk = recommended_num_knots(incl, NUM_KNOTS)

    # ── Rebuild Optimizer from scipy best-fit (warm start) ────────────────────
    fit_disk = BASE_DISK.copy()
    fit_disk["inclination"] = incl

    spf_params = InterpolatedUnivariateSpline_SPF.params.copy()
    spf_params["num_knots"]          = nk
    spf_params["knot_values"]        = jnp.array(r["knot_values"])
    spf_params["forwardscatt_bound"] = r["fwd_bound"]
    spf_params["backscatt_bound"]    = r["back_bound"]

    fit_misc = BASE_MISC.copy()
    fit_misc["flux_scaling"] = r["flux_scaling"]

    opt = Optimizer(
        ScatteredLightDisk, DustEllipticalDistribution2PowerLaws,
        InterpolatedUnivariateSpline_SPF, None,
        fit_disk, spf_params, None, fit_misc,
    )
    # Each inclination gets its own HDF5 backend so chains can be resumed
    opt.name = os.path.join(MCMC_DIR, f"spline_incl{int(incl):02d}")

    # ── Bounds ────────────────────────────────────────────────────────────────
    mcmc_bounds = (
        [np.full(nk, 1e-6), 1e2],
        [np.full(nk, 1e3),  1e12],
    )

    # ── Run MCMC ──────────────────────────────────────────────────────────────
    # confirm_overwrite=False suppresses the interactive prompt (notebook context)
    mc_model = opt.mcmc(
        fit_keys=MCMC_KEYS,
        logscaled_params=MCMC_LOGSCALED,
        array_params=MCMC_ARRAYS,
        target_image=r["truth_img"],
        err_map=r["err_map"],
        BOUNDS=mcmc_bounds,
        nwalkers=NWALKERS, niter=NITER, burns=BURNS,
        confirm_overwrite=False,
    )
    print(f"  Done. Posterior samples: {mc_model._get_flatchain(scaled=True).shape[0]}")

    # ── Compute posterior SPF envelope ────────────────────────────────────────
    flatchain  = mc_model._get_flatchain(scaled=True)   # (n_samples, nk+1)
    kv_samples = flatchain[:, :nk]                       # free knot values

    knots = InterpolatedUnivariateSpline_SPF.get_knots(spf_params)

    # Subsample to at most 2000 curves for speed; retain statistical coverage
    stride = max(1, len(kv_samples) // 2000)
    spf_samples = []
    for kv in kv_samples[::stride]:
        spline = InterpolatedUnivariateSpline_SPF.pack_pars(jnp.array(kv), knots=knots)
        curve  = np.array(InterpolatedUnivariateSpline_SPF.compute_phase_function_from_cosphi(
            spline, jnp.array(cos_grid)
        ))
        spf_samples.append(curve)
    spf_samples = np.array(spf_samples)          # (n_sub, n_angles)

    spf_median = np.median(spf_samples, axis=0)
    spf_lo     = np.percentile(spf_samples, 16, axis=0)   # −1σ
    spf_hi     = np.percentile(spf_samples, 84, axis=0)   # +1σ

    cp_back, cp_fwd = probed_cosphi_range(incl)
    in_range = (cos_grid >= cp_back) & (cos_grid <= cp_fwd)
    frac_err_median = float(np.max(
        np.abs(spf_median[in_range] - r["true_spf_norm"][in_range])
        / (np.abs(r["true_spf_norm"][in_range]) + 1e-30)
    ))
    print(f"  Max SPF error (median, probed range): {frac_err_median*100:.2f}%")

    mcmc_results[incl] = dict(
        mc_model=mc_model,
        spf_samples=spf_samples,
        spf_median=spf_median,
        spf_lo=spf_lo,
        spf_hi=spf_hi,
        true_spf_norm=r["true_spf_norm"],
        recovered_spf=r["recovered_spf"],   # scipy best-fit for comparison
        cp_fwd=cp_fwd, cp_back=cp_back,
        max_frac_err_median=frac_err_median,
    )

### Convergence diagnostics

Before interpreting the posterior, verify that the chains are well-converged.
The rule of thumb for emcee is **NITER ≥ 50 × τ**, where τ is the integrated
autocorrelation time. If the recommended NITER below exceeds the value used
above, re-run with the larger value (set `resume=False` to start fresh).

In [ ]:
# Knot positions are the same for all inclinations — compute once
_spf_tmp = InterpolatedUnivariateSpline_SPF.params.copy()
_spf_tmp["num_knots"] = NUM_KNOTS
_knots_full = np.array(InterpolatedUnivariateSpline_SPF.get_knots(_spf_tmp))
# Free knot cos_phi positions (drop the fixed center at index NUM_KNOTS//2)
FREE_KNOT_COS = np.delete(_knots_full, NUM_KNOTS // 2)   # shape (NUM_KNOTS,)
FLUX_DIM = NUM_KNOTS                                       # last chain dimension

param_labels = [f"k{i} ({np.degrees(np.arccos(c)):.0f}°)" for i, c in enumerate(FREE_KNOT_COS)] + ["flux"]

print(f"{'incl':>6}  {'probed range':>20}  {'constrained dims':>16}  "
      f"{'max_tau':>8}  {'niter_used':>10}  {'niter_rec':>10}  {'converged':>9}")
print("-" * 95)

for incl in INCLINATIONS:
    mc_model = mcmc_results[incl]["mc_model"]
    cp_back, cp_fwd = probed_cosphi_range(incl)
    ang_fwd  = np.degrees(np.arccos(cp_fwd))
    ang_back = np.degrees(np.arccos(cp_back))

    # Which free knots fall within the probed cos_phi range?
    knot_in_range = (FREE_KNOT_COS >= cp_back) & (FREE_KNOT_COS <= cp_fwd)
    constrained_dims = list(np.where(knot_in_range)[0]) + [FLUX_DIM]

    try:
        tau_all = mc_model.sampler.get_autocorr_time(quiet=True)   # (ndim,)
        tau_constrained = tau_all[constrained_dims]
        max_tau = float(np.max(tau_constrained))
    except Exception:
        tau_all = np.full(NUM_KNOTS + 1, np.nan)
        tau_constrained = np.array([np.nan])
        max_tau = np.nan

    niter_rec = int(np.ceil(50 * max_tau)) if np.isfinite(max_tau) else -1
    converged = NITER >= niter_rec if niter_rec > 0 else False

    print(f"{incl:>6.0f}  {ang_fwd:>8.1f}–{ang_back:<8.1f}°  "
          f"{len(constrained_dims):>16d}  "
          f"{max_tau:>8.1f}  {NITER:>10d}  {niter_rec:>10d}  "
          f"{'yes' if converged else 'NO':>9}")

    for dim in constrained_dims:
        print(f"         {param_labels[dim]}: τ = {tau_all[dim]:.1f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("MCMC posterior SPF recovery", fontsize=14)

for ax, incl in zip(axes.flat, INCLINATIONS):
    m = mcmc_results[incl]
    cp_back, cp_fwd = m["cp_back"], m["cp_fwd"]
    ang_fwd  = float(np.degrees(np.arccos(cp_fwd)))
    ang_back = float(np.degrees(np.arccos(cp_back)))

    # ±1σ posterior envelope
    ax.fill_between(
        angles, m["spf_lo"], m["spf_hi"],
        alpha=0.35, color="C0", label="Posterior ±1σ",
    )
    ax.plot(angles, m["spf_median"], "C0-", lw=1.8, label="Posterior median")
    ax.plot(angles, m["recovered_spf"], "C1--", lw=1.5, label="Scipy best-fit")
    ax.plot(angles, m["true_spf_norm"], "k-", lw=2, label="True HG")

    ax.axvspan(ang_fwd, ang_back, alpha=0.10, color="grey",
               label=f"Probed ({ang_fwd:.0f}–{ang_back:.0f}°)")
    ax.axvline(90, color="grey", ls=":", lw=0.8)
    ax.axhline(1.0, color="grey", ls=":", lw=0.8)

    ax.set_xlim(0, 180)
    ax.set_xlabel("Scattering angle (deg)")
    ax.set_ylabel("Normalised SPF")
    ax.set_title(f"incl = {incl}°  (median err = {m['max_frac_err_median']*100:.2f}%)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "mcmc_spf_posterior_summary.png"), dpi=150)
plt.show()

### Disk model diagnostics

Reconstruct a model image from the MCMC posterior median and compare it to the injected truth image. This mirrors the per-inclination disk comparison done for the scipy fit above.

In [ ]:
# ── MCMC disk model comparison ────────────────────────────────────────────
# For each inclination, reconstruct a model image from the posterior median
# knot values and compare to the injected truth image.

for incl in INCLINATIONS:
    m  = mcmc_results[incl]
    r  = results[incl]
    nk = recommended_num_knots(incl, NUM_KNOTS)

    # Posterior median knot values (real-space, already unlogged by opt.mcmc)
    flatchain   = m["mc_model"]._get_flatchain(scaled=True)  # (n_samples, nk+1)
    median_kv   = np.median(flatchain[:, :nk], axis=0)        # (nk,)
    median_flux = float(np.median(flatchain[:, nk]))

    # Rebuild optimizer at the posterior median
    fit_disk = BASE_DISK.copy()
    fit_disk["inclination"] = incl

    spf_params_med = InterpolatedUnivariateSpline_SPF.params.copy()
    spf_params_med["num_knots"]          = nk
    spf_params_med["knot_values"]        = jnp.array(median_kv)
    spf_params_med["forwardscatt_bound"] = r["fwd_bound"]
    spf_params_med["backscatt_bound"]    = r["back_bound"]

    fit_misc_med = BASE_MISC.copy()
    fit_misc_med["flux_scaling"] = median_flux

    opt_med = Optimizer(
        ScatteredLightDisk, DustEllipticalDistribution2PowerLaws,
        InterpolatedUnivariateSpline_SPF, None,
        fit_disk, spf_params_med, None, fit_misc_med,
    )
    mcmc_median_img = np.array(opt_med.get_model())

    # ── 3-panel: truth | MCMC median | residual ───────────────────────────
    residual = r["truth_img"] - mcmc_median_img
    vmin = np.percentile(r["truth_img"][r["truth_img"] > 0], 5)
    vmax = r["truth_img"].max()
    res_lim = np.max(np.abs(residual))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"MCMC posterior median  —  incl = {incl}°", fontsize=13)

    im0 = axes[0].imshow(r["truth_img"],  cmap="inferno", origin="lower", vmin=vmin, vmax=vmax)
    axes[0].set_title("Truth (HG SPF)")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(mcmc_median_img, cmap="inferno", origin="lower", vmin=vmin, vmax=vmax)
    axes[1].set_title("MCMC median model")
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(residual, cmap="seismic", origin="lower", vmin=-res_lim, vmax=res_lim)
    axes[2].set_title("Residual (Truth − MCMC median)")
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"mcmc_disk_comparison_incl{int(incl):02d}.png"), dpi=150)
    plt.show()

    # ── Per-inclination SPF comparison ────────────────────────────────────
    cp_back, cp_fwd = m["cp_back"], m["cp_fwd"]
    ang_fwd  = float(np.degrees(np.arccos(cp_fwd)))
    ang_back = float(np.degrees(np.arccos(cp_back)))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(angles, m["spf_lo"], m["spf_hi"],
                    alpha=0.35, color="C0", label="Posterior ±1σ")
    ax.plot(angles, m["spf_median"],    "C0-",  lw=1.8, label="Posterior median")
    ax.plot(angles, m["recovered_spf"], "C1--", lw=1.5, label="Scipy best-fit")
    ax.plot(angles, m["true_spf_norm"], "k-",   lw=2.0, label="True HG")
    ax.axvspan(ang_fwd, ang_back, alpha=0.10, color="grey",
               label=f"Probed ({ang_fwd:.0f}–{ang_back:.0f}°)")
    ax.axvline(90,  color="grey", ls=":", lw=0.8)
    ax.axhline(1.0, color="grey", ls=":", lw=0.8)
    ax.set_xlim(0, 180)
    ax.set_xlabel("Scattering angle (deg)")
    ax.set_ylabel("Normalised SPF")
    ax.set_title(f"MCMC SPF recovery  incl={incl}°  (median err={m['max_frac_err_median']*100:.2f}%)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"mcmc_spf_comparison_incl{int(incl):02d}.png"), dpi=150)
    plt.show()
